**7.- Entrena y perfecciona un árbol de decisión para el conjunto de datos moons siguiendo estos pasos:**

Utiliza make_moons(n_samples=10000, noise=0.4) para generar un conjunto de datos moons

In [10]:
from sklearn.datasets import make_moons

# Fijamos random_state para que el resultado sea reproducible.
X, y = make_moons(n_samples=10_000, noise=0.4, random_state=42)

X.shape, y.shape


((10000, 2), (10000,))

Usa train_test_split() para dividir el conjunto de datos en un conjunto de entrenamiento y otro de prueba.

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape


((8000, 2), (2000, 2))

Utiliza una búsqueda exhaustiva con validación cruzada ( con ayuda de la clase GridSearchCV) para encontrar buenos valores de hiperparametros para un DescisionTreeClassifier. Pista: prubea varios valores para max_leaf_nodes

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

param_grid = {"max_leaf_nodes": [2, 5, 10, 20, 30, 40, 50, 75, 100], "min_samples_split": [2, 3, 4]}
grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Mejores hiperparámetros:", grid_search.best_params_)
print(f"Precisión media de validación cruzada: {grid_search.best_score_:.4f}")


Mejores hiperparámetros: {'max_leaf_nodes': 30, 'min_samples_split': 2}
Precisión media de validación cruzada: 0.8602


Entrénalo con el conjunto de entrenameinto completo usando estos hiperparametros y mide el rendimiento de tu modelo con el conjunto de prueba. Deberías conseguir una precisión de 85 % - 87 % aproximadamente

In [13]:
from sklearn.metrics import accuracy_score

# GridSearchCV reajusta el mejor modelo con todo el conjunto de entrenamiento.
best_tree = grid_search.best_estimator_
y_pred = best_tree.predict(X_test)
tree_accuracy = accuracy_score(y_test, y_pred)

print(f"Precisión del árbol en el conjunto de prueba: {tree_accuracy:.4f}")


Precisión del árbol en el conjunto de prueba: 0.8660


**8.- Cultiva un bosque siguiendo estos pasos:**

Siguiendo con el ejercicio anterior, genera 1.000 subconjuntos del conjunto de entrenamiento, cada uno con 100 instancias elegidas aleatoriamente. Pista: puedes usar la clase ShuffleSplit de Scikit-Learn para esto.

In [14]:
from sklearn.model_selection import ShuffleSplit

n_trees = 1_000
n_instances = 100
mini_sets = []
splitter = ShuffleSplit(n_splits=n_trees, train_size=n_instances, random_state=42)

for mini_train_indices, _ in splitter.split(X_train):
    X_mini_train = X_train[mini_train_indices]
    y_mini_train = y_train[mini_train_indices]
    mini_sets.append((X_mini_train, y_mini_train))

len(mini_sets), mini_sets[0][0].shape


(1000, (100, 2))

Entrena un árbol de decisión con cada subconjunto, utilizando los mejores valores de hiperparametros hallados en el ejercicio anterior. Evalua estos 1000 árboles de decisión con el conjunto de prueba. Dado que se entrenaron con conjuntos de datos más pequeños, es probable que estos arboles de decisión tengan peor rendimiento que el primero, consiguiendo solo un 80% de precisión, más o menos.

In [15]:
from sklearn.base import clone
import numpy as np

forest = [clone(best_tree) for _ in range(n_trees)]
tree_accuracies = []

for tree, (X_mini_train, y_mini_train) in zip(forest, mini_sets):
    tree.fit(X_mini_train, y_mini_train)
    tree_predictions = tree.predict(X_test)
    tree_accuracies.append(accuracy_score(y_test, tree_predictions))

print(f"Precisión media de los 1.000 árboles: {np.mean(tree_accuracies):.4f}")
print(f"Mejor precisión individual: {np.max(tree_accuracies):.4f}")


Precisión media de los 1.000 árboles: 0.7955
Mejor precisión individual: 0.8585


Ahora viene lo bueno. Para cada instancia del conjunto de prueba, genera las predicciones de los mil arboles de decisión y conserva solo la predicción mas frecuente ( puedes usar la función mode de SciPy para esto). Este enfoque dte da predicciones del voto de la mayoria sobre el conjunto de prueba.

In [16]:
from scipy.stats import mode

all_predictions = np.array([tree.predict(X_test) for tree in forest])
y_pred_majority_votes = mode(all_predictions, axis=0, keepdims=False).mode

all_predictions.shape, y_pred_majority_votes.shape


((1000, 2000), (2000,))

Evalúa estas predicciones con el conjunto de prueba: deberías obtener una precisión ligeramente más alta que con el primer modelo ( aproximadamente un 0,5 - 1,5 % mas alta). ¡Felicidiades, has entrenado un clasificador random forest!

In [17]:
forest_accuracy = accuracy_score(y_test, y_pred_majority_votes)
improvement = forest_accuracy - tree_accuracy

print(f"Precisión del árbol individual: {tree_accuracy:.4f}")
print(f"Precisión del bosque por voto mayoritario: {forest_accuracy:.4f}")
print(f"Mejora absoluta: {improvement:.4f} ({improvement * 100:.2f} puntos porcentuales)")


Precisión del árbol individual: 0.8660
Precisión del bosque por voto mayoritario: 0.8630
Mejora absoluta: -0.0030 (-0.30 puntos porcentuales)
